# NeOlap mit Python: das Modell `tiny`

Dieses Notebook läuft **unverändert** auf einer lokalen Datei und gegen einen Server. Nur die erste
Code-Zelle unterscheidet sich. Alles danach ist dieselbe API.

Voraussetzungen: `pip install neolap[pandas]` und die Kommandozeile `neolap` (für das Demo-Modell).


## 1. Verbinden

Entweder eine Datei öffnen (exklusiv, kein Server nötig) oder sich mit einem Server verbinden.

In [ ]:
import os, subprocess, tempfile, pathlib
import neolap

MODE = os.environ.get("NEOLAP_NOTEBOOK_MODE", "local")   # "local" oder "remote"

workdir = pathlib.Path(tempfile.mkdtemp())
if MODE == "local":
    path = workdir / "tiny.neolap"
    subprocess.run(["neolap", "demo", "tiny", str(path)], check=True, capture_output=True)
    db = neolap.open(path, user="notebook")          # eine Datei, exklusiv
else:
    # Server-URL und Token kommen aus der Umgebung; ein Token legt ein Admin an (`neolap user token`)
    db = neolap.connect(os.environ["NEOLAP_URL"], token=os.environ["NEOLAP_TOKEN"])

print(db.cubes)
cube = db.cube("sales")
print(cube)

## 2. Zellen lesen

Eine Adresse hat ein Element je Dimension, in der Reihenfolge von `cube.dimensions`. Aggregate und Regeln werden mitgerechnet.

In [ ]:
print("Basiszelle   :", cube["Jan", "North", "sedan", "Sales"])
print("Quartal       :", cube["Q1", "Total", "All", "Sales"])
print("Jahr, Gewinn  :", cube["Year", "Total", "All", "Profit"])
print("Marge (Regel) :", round(cube["Year", "Total", "All", "Profit in %"], 4))
print("leere Zelle   :", cube.get(("Year", "Total", "All", "Profit in %"), rules=False))

## 3. Bereiche und pandas

`cube.area(...)` wählt Zellen aus und liest sie erst, wenn man sie braucht. Die Dimensionen werden zu `category`, die Werte zu `float64`.

In [ ]:
df = cube.area(years=["Jan", "Feb", "Mar"], regions="North", products="sedan").to_pandas()
df

In [ ]:
# breit: die Kennzahlen als Spalten, mit Aggregaten
wide = cube.area(years=["Q1", "Q2"], regions="Total", products="All", wide="measures", aggregates=True).to_pandas()
wide

In [ ]:
# mit Pandas weiterrechnen
q = cube.area(years=["Q1", "Q2", "Q3", "Q4"], regions=["North", "South"], products="All", measures="Sales", aggregates=True).to_pandas()
q.pivot_table(index="years", columns="regions", values="value", observed=True)

## 4. Schreiben

Eine Basiszelle wird direkt gesetzt. Auf einem Aggregat wird der Wert auf die Basiszellen **verteilt** (Splash).

In [ ]:
before = cube["Q1", "Total", "All", "Sales"]
cube["Jan", "North", "sedan", "Sales"] = 2000          # Basiszelle (vorher 1000)
print("Quartal vorher/nachher:", before, "->", cube["Q1", "Total", "All", "Sales"])

In [ ]:
changed = cube.write(("Q1", "Total", "All", "Sales"), 100_000)   # proportional verteilt
print(changed, "Zellen geändert; Quartal jetzt", cube["Q1", "Total", "All", "Sales"])
cube.write(("Q1", "Total", "All", "Sales"), "#24000")              # gleichmäßig auf alle 24 Zellen
print("jede Zelle:", cube["Jan", "North", "sedan", "Sales"])
cube.write(("Q1", "Total", "All", "Sales"), "+10%")                 # alle um 10 % erhöhen
print("Quartal +10 %:", round(cube["Q1", "Total", "All", "Sales"], 2))

### Rückgängig

Jeder Schreibvorgang lässt sich zurücknehmen, solange niemand sonst die Zellen geändert hat.

In [ ]:
for _ in range(4):
    cube.undo()
print("wieder wie am Anfang:", cube["Q1", "Total", "All", "Sales"])

## 5. Mehrere Nutzer: Konflikte erkennen

Mit `expected_generation` schlägt ein Schreibvorgang fehl, wenn jemand eine betroffene Zelle seit dem Lesen geändert hat.
Die Ausnahme sagt, wer es war.

In [ ]:
seen = cube.generation                                  # Stand, den "wir" gesehen haben
cube["Feb", "North", "sedan", "Sales"] = 1                # eine Änderung dazwischen (hier von uns selbst)
try:
    cube.write(("Q1", "North", "sedan", "Sales"), 500, expected_generation=seen)
except neolap.ConflictError as e:
    print(f"Konflikt: {e.user} hat {e.cell} auf {e.value} gesetzt")
cube.write(("Q1", "North", "sedan", "Sales"), 500, expected_generation=cube.generation)   # mit dem aktuellen Stand geht es
cube.undo(); cube.undo()

## 6. Daten laden

`cube.load(...)` nimmt einen `DataFrame` oder eine Datei und lädt sie **als eine Operation**: alles oder nichts, in einem Schritt rückgängig zu machen.
Mit `replace_area` ergibt zweimaliges Laden dasselbe wie einmaliges. Eine Vorschau (`dry_run=True`) ändert nichts.

In [ ]:
import pandas as pd
plan = pd.DataFrame({
    "Monat":   ["Jan", "Feb", "Mar"],
    "Region":  ["North"] * 3,
    "Produkt": ["sedan"] * 3,
    "Umsatz":  [1234.5, 2000.0, 3100.25],
    "Kosten":  [800.0, 1300.0, 2000.0],
})
cols = dict(mapping={"years": "Monat", "regions": "Region", "products": "Produkt"},
            values={"Sales": "Umsatz", "Cost": "Kosten"})

preview = cube.load(plan, dry_run=True, **cols)
print("Vorschau :", {k: preview[k] for k in ("rows_read", "cells_updated", "cells_created", "dry_run")})
report = cube.load(plan, mode="replace_area", **cols)
print("geladen  :", {k: report[k] for k in ("rows_read", "cells_updated", "cells_created")})
print("Jan/Sales:", cube["Jan", "North", "sedan", "Sales"])
again = cube.load(plan, mode="replace_area", **cols)
print("nochmal  :", {k: again[k] for k in ("cells_created", "cells_updated", "cells_deleted")}, "(nichts geändert)")

In [ ]:
# eine fehlerhafte Zeile stoppt alles und nennt die Zeile
bad = plan.copy(); bad.loc[1, "Monat"] = "Smarch"
try:
    cube.load(bad, **cols)
except neolap.InputError as e:
    print("abgelehnt:", e)
print("unverändert:", cube["Feb", "North", "sedan", "Sales"])

## 7. Aufräumen

In [ ]:
db.close()   # gibt die Datei frei (bei einem Server: nichts zu tun)
print("fertig")